In [ ]:
%load_ext autoreload
%autoreload 2

import os
from src.constants import BASEDIR
os.chdir(BASEDIR)

from src.data.objects import Protein, ProteinDocument
from src.data.preprocessing import load_named_preprocessor
from src.sequence.utils import sequence_identity
from src.evaluators.esmfold import esmfold_output_to_proteins
from src.structure.superimposition import tm_score, rmsd
from src.models.inference import InterleavedInverseFoldingPromptBuilder, ProFamSampler
from src.models.utils import load_named_model

import py3Dmol
from transformers import AutoTokenizer, EsmForProteinFolding

In [ ]:
# BB_NAME = "afdb/A0A1F9XLJ6"  # I've added some high plddt structures to example_data
# BB_NAME = "afdb/A0A7H4PFQ4"
# BB_NAME = "afdb/A0A7R9Z4H1"
BB_NAME = "1qys"  # top7
MODEL_NAME = "twilight-snowflake"
MODEL_CONFIG = "llama_medium"

In [ ]:
protein = Protein.from_pdb(os.path.join(BASEDIR, f"data/example_data/backbones/{BB_NAME}.pdb"), bfactor_is_plddt=BB_NAME.startswith("afdb"))
protein.sequence

In [ ]:
view = py3Dmol.view(width=450, height=400)
protein.view(view)
view.zoomTo()
view.show()

TODO: figure out an easy way to instantiate preprocessor

TODO: if using random transforms - allow applying the pipeline multiple times to single target

In [ ]:
preprocessor = load_named_preprocessor("parquet_structure", overrides=["structure_tokens_col=null"])

In [ ]:
preprocessor.transform_fns

In [ ]:
model = load_named_model(MODEL_CONFIG, overrides=["model.embed_coords=True"])

In [ ]:
prompt_builder = InterleavedInverseFoldingPromptBuilder(
    preprocessor=preprocessor,
    max_tokens=1536,
)

In [ ]:
prompt = prompt_builder(ProteinDocument.from_proteins([protein], representative_accession=protein.accession), model.tokenizer)

Prompts are randomly rotated.

In [ ]:
prompt_protein = prompt[0].slice_arrays(slice(0, len(protein.sequence)))
prompt_protein.backbone_coords = prompt_protein.backbone_coords * 6
view = py3Dmol.view(width=450, height=400)
prompt_protein.view(view)
view.zoomTo()
view.show()

In [ ]:
prompt.sequences

In [ ]:
sampler = ProFamSampler(
    "if_v1",
    model,
    prompt_builder=prompt_builder,
    document_token="[AFDB]",
    sampling_kwargs={"batch_size": 3, "temperature": 0.1},
    checkpoint_path=os.path.join(BASEDIR, f"outputs/inverse_folding/{MODEL_NAME}/checkpoints/last.ckpt"),
    match_representative_length=True,
)

In [ ]:
samples, prompt = sampler.sample_seqs(
    ProteinDocument.from_proteins(
        [protein], representative_accession=protein.accession
    ), num_samples=3
)

In [ ]:
[sequence_identity(s, protein.sequence) for s in samples]

For example of using HF ESMFold, see:
https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/protein_folding.ipynb

HF ESMFold has fewer dependencies than ESM ESMFold

In [ ]:
esmfold = EsmForProteinFolding.from_pretrained(
    "facebook/esmfold_v1"
).eval()
esmfold.float()
tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")

In [ ]:
sample_out = esmfold.infer(samples[1])

In [ ]:
sample_prot = esmfold_output_to_proteins(sample_out)[0]
sample_prot.plddt.mean()

In [ ]:
sample_prot.sequence, samples[1]

In [ ]:
view = py3Dmol.view(width=450, height=400)
protein.view_superimposed(view, sample_prot)
view.zoomTo()
view.show()

In [ ]:
tm_score(protein, sample_prot), rmsd(protein, sample_prot)

In [ ]:
ref_out = esmfold.infer(protein.sequence)

In [ ]:
ref_prot = esmfold_output_to_proteins(ref_out)[0]
ref_prot.plddt.mean()

In [ ]:
view = py3Dmol.view(width=450, height=400)
protein.view_superimposed(view, ref_prot)
view.zoomTo()
view.show()

In [ ]:
tm_score(protein, ref_prot), rmsd(protein, ref_prot)